# Thu thập đa tệp và bóc tách cấu trúc phân cấp (Hierarchical Parsing)

> **Môn học:** Nhập môn Khoa học Dữ liệu — HCMUS  
> **Milestone 2:** Bóc tách Cấu trúc Tài liệu & Chuẩn Hóa Phân Cấp  
> **Thành viên phụ trách:** Đặng Ngọc Hoàng (Bộ bóc tách & Cây phân cấp), Dương Ngọc Minh Thư (Thu thập đường biên dịch), Nguyễn Lê Anh Tuấn (Tổng hợp & Trực quan hóa dữ liệu)  
> **Mục tiêu:** Bóc tách cấu trúc phân cấp từ toàn văn mã nguồn LaTeX, xử lý đa tệp đệ quy, nhận diện cấp độ đề mục động, ngắt câu an toàn ngữ cảnh học thuật và thống kê thực nghiệm trên toàn bộ 5.000 bài báo (15.723 phiên bản).

---

### Tổng quan phương pháp tiếp cận

Trong kho lưu trữ mở arXiv, mã nguồn LaTeX có cấu trúc phân mảnh cao: hơn 70% bài báo chia nhỏ nội dung vào nhiều tệp `.tex` con thông qua các lệnh `\input`, `\include`, `\subfile`. Để tái hiện chính xác cấu trúc phân cấp, quy trình phân tích được thiết kế theo kiến trúc đường ống 3 tầng:
1. **Tầng 1 (Gather):** Quét đệ quy cấu trúc thư mục, định vị tệp biên dịch chính (`\documentclass`), chấm điểm ưu tiên ứng viên và thiết lập đường biên dịch thống nhất (`compile_path`).
2. **Tầng 2 (Hierarchy & Sentence Splitting):** Phân tích ngữ pháp cây phân cấp, suy luận thứ bậc đề mục theo thứ tự xuất hiện, và ngắt câu dựa trên cơ chế che chắn ký tự (masking) để bảo vệ công thức toán và từ viết tắt.
3. **Tầng 3 (Store & Dedup):** Sinh bảng `elements` lưu trữ các khối nội dung duy nhất và bảng `hierarchy` ánh xạ quan hệ phân cấp nút-cha cho từng phiên bản.

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

data_dir = Path("../data") if Path("../data").exists() else Path("data")

# Nạp nhật ký bóc tách toàn diện 5.000 bài báo từ Đặng Ngọc Hoàng
parse_path = data_dir / "parse.csv"
if not parse_path.exists():
    parse_path = data_dir / "hoang_m2_parse.csv"

df_parse = pd.read_csv(parse_path)

# Nạp mẫu cấu trúc elements và hierarchy
df_elem_sample = pd.read_csv(data_dir / "elements_sample.csv") if (data_dir / "elements_sample.csv").exists() else None
df_hier_sample = pd.read_csv(data_dir / "hierarchy_sample.csv") if (data_dir / "hierarchy_sample.csv").exists() else None

n_total_versions = len(df_parse)
n_unique_pubs = df_parse['arxiv_id'].nunique()
status_counts = df_parse['status'].value_counts()
total_elements = df_parse['n_elements'].sum()
total_sentences = df_parse['n_sentences'].sum()
total_formulas = df_parse['n_formulas'].sum()
total_figures = df_parse['n_figures'].sum()

print("===============================================================================")
print("TỔNG KẾT DỮ LIỆU BÓC TÁCH CẤU TRÚC PHÂN CẤP (MILESTONE 2):")
print(f"- Tổng số bài báo nghiên cứu (Publications) : {n_unique_pubs:,} bài")
print(f"- Tổng số phiên bản đã bóc tách (Versions) : {n_total_versions:,} phiên bản")
print(f"- Tỷ lệ phiên bản thành công (ok)           : {status_counts.get('ok', 0):,} ({status_counts.get('ok', 0)/n_total_versions*100:.2f}%)")
print(f"- Tỷ lệ phiên bản rỗng (empty - PDF/no TeX) : {status_counts.get('empty', 0):,} ({status_counts.get('empty', 0)/n_total_versions*100:.2f}%)")
print(f"- Tỷ lệ lỗi (fail)                          : {status_counts.get('fail', 0):,} ({status_counts.get('fail', 0)/n_total_versions*100:.2f}%)")
print(f"- TỔNG SỐ PHẦN TỬ BÓC TÁCH (ELEMENTS)       : {total_elements:,} phần tử")
print(f"  + Câu văn (Sentences)                     : {total_sentences:,} ({total_sentences/total_elements*100:.1f}%)")
print(f"  + Công thức toán học (Formulas)           : {total_formulas:,} ({total_formulas/total_elements*100:.1f}%)")
print(f"  + Hình ảnh & Bảng biểu (Figures/Tables)   : {total_figures:,} ({total_figures/total_elements*100:.1f}%)")
print("===============================================================================")

TỔNG KẾT DỮ LIỆU BÓC TÁCH CẤU TRÚC PHÂN CẤP (MILESTONE 2):
- Tổng số bài báo nghiên cứu (Publications) : 5,000 bài
- Tổng số phiên bản đã bóc tách (Versions) : 15,723 phiên bản
- Tỷ lệ phiên bản thành công (ok)           : 14,658 (93.23%)
- Tỷ lệ phiên bản rỗng (empty - PDF/no TeX) : 1,022 (6.50%)
- Tỷ lệ lỗi (fail)                          : 22 (0.14%)
- TỔNG SỐ PHẦN TỬ BÓC TÁCH (ELEMENTS)       : 6,380,940 phần tử
  + Câu văn (Sentences)                     : 5,038,034 (79.0%)
  + Công thức toán học (Formulas)           : 596,888 (9.4%)
  + Hình ảnh & Bảng biểu (Figures/Tables)   : 150,990 (2.4%)


### 1. Thuật toán thu thập đường biên dịch và giải quyết phụ thuộc đa tệp

Module `src/parser/gather.py` do **Dương Ngọc Minh Thư** phụ trách đảm nhiệm việc hợp nhất mã nguồn phân mảnh:
- **Nhận diện tệp gốc (`root_file`):** Quét toàn bộ tệp `.tex` trong thư mục phiên bản để tìm lệnh khai báo `\documentclass`. Trong trường hợp xuất hiện nhiều ứng viên (ví dụ tài liệu chứa cả tệp mẫu của tạp chí hoặc tệp phụ trợ), thuật toán áp dụng hàm chấm điểm ưu tiên:
  + Cộng điểm cho tệp chứa `\begin{document}` và `\end{document}`.
  + Cộng điểm cho tên tệp tiêu chuẩn (`main.tex`, `ms.tex`, tên trùng với arXiv ID).
  + Trừ điểm nặng cho tệp nằm trong thư mục con `tikz/`, `figures/` hoặc mang tên `template.tex`.
- **Duyệt đệ quy và chống vòng lặp:** Khởi đầu từ tệp gốc, thuật toán bóc tách các lệnh nạp tệp: `\input{path}`, `\include{path}`, `\subfile{path}`. Để ngăn ngừa vòng lặp vô hạn do các tác giả nạp chéo lẫn nhau, hệ thống sử dụng tập băm `visited_set` lưu trữ đường dẫn chuẩn hóa tuyệt đối. Toàn bộ các tệp hợp lệ được ghép nối thành chuỗi văn bản tuần tự duy nhất theo đúng thứ tự biên dịch của LaTeX.

In [2]:
# Biểu đồ tương tác Plotly: Phân bổ trạng thái xử lý 15.723 phiên bản
status_df = pd.DataFrame({
    "Trạng thái": ["Thành công (ok)", "Rỗng hợp lệ (empty)", "Thất bại (fail)", "Bỏ qua (skipped)"],
    "Số lượng": [
        status_counts.get("ok", 0),
        status_counts.get("empty", 0),
        status_counts.get("fail", 0),
        status_counts.get("skipped", 0)
    ]
})

fig_status = px.pie(
    status_df, 
    names="Trạng thái", 
    values="Số lượng",
    hole=0.45,
    color="Trạng thái",
    color_discrete_map={
        "Thành công (ok)": "#2ca02c",
        "Rỗng hợp lệ (empty)": "#ff7f0e",
        "Thất bại (fail)": "#d62728",
        "Bỏ qua (skipped)": "#7f7f7f"
    },
    title="Phân bổ trạng thái bóc tách trên toàn bộ 15.723 phiên bản (Đặng Ngọc Hoàng)"
)
fig_status.update_traces(textposition='inside', textinfo='percent+label', hoverinfo='label+value+percent')
fig_status.update_layout(height=420, margin=dict(t=50, b=20, l=20, r=20))
fig_status.show()

### 2. Xây dựng cây phân cấp và cơ chế tự suy cấp độ đề mục

Đề bài đặt ra yêu cầu nghiêm ngặt: *"Since the exact nesting order depends on the publication's specific formatting, the parser is required to **automatically identify** the appropriate hierarchical level for each such element."*

Module `src/parser/hierarchy.py` do **Đặng Ngọc Hoàng** phụ trách giải quyết bài toán này với các nguyên tắc kiến trúc sau:
1. **Suy luận động cấp bậc đề mục (Không hardcode):**
   Không thể mặc định `\section` luôn là cấp 1. Các bài báo định dạng `book` hoặc `report` có lệnh `\chapter` đứng trên `\section`; ngược lại bài dùng `article` không có `\chapter`. Thuật toán duyệt qua tài liệu và gán cấp bậc theo **thứ tự xuất hiện lần đầu** của từng lệnh phân mục:
   ```python
   SECTION_CMD = re.compile(r"\\(chapter|section|subsection|subsubsection|paragraph|subparagraph)\*?\s*\{")
   ```
   Đặc biệt, biểu thức chính quy bắt trọn dấu sao `*` để thu nạp các phân mục không đánh số như `\section*{Acknowledgements}` và `\section*{Appendix}` theo đúng yêu cầu bắt buộc của đề bài.
2. **Cắt bỏ phần References nhưng bảo tồn Appendix:**
   Phần tài liệu tham khảo (`thebibliography`) không được đưa vào cây phân cấp theo quy định của đề bài. Tuy nhiên, nhiều bài báo đặt phần Phụ lục (Appendix) nằm **sau** References. Thuật toán cắt bỏ chính xác từ điểm bắt đầu References đến hết môi trường `thebibliography` hoặc lệnh section kế tiếp, sau đó nối liền phần Appendix phía sau vào thân cây.
3. **Môi trường liệt kê (`itemize` & `enumerate`):**
   Đúng theo đặc tả: một khối `\begin{itemize}` đóng vai trò là phần tử cha, mỗi điểm `\item` là một phần tử con cấp kế tiếp. Các câu văn bên trong `\item` tiếp tục được tách thành các phần tử lá.

In [3]:
# Biểu đồ tương tác Plotly: Phân bố độ sâu cây phân cấp (max_depth)
depth_counts = df_parse[df_parse['status'] == 'ok']['max_depth'].value_counts().sort_index().reset_index()
depth_counts.columns = ['Độ sâu cây (max_depth)', 'Số phiên bản']

fig_depth = px.bar(
    depth_counts,
    x='Độ sâu cây (max_depth)',
    y='Số phiên bản',
    text='Số phiên bản',
    title='Phân bố độ sâu tối đa của cây phân cấp trên 14.658 phiên bản thành công',
    color='Số phiên bản',
    color_continuous_scale='Viridis'
)
fig_depth.update_traces(texttemplate='%{text:,}', textposition='outside')
fig_depth.update_layout(xaxis=dict(tickmode='linear', dtick=1), height=450, margin=dict(t=50, b=40, l=40, r=20))
fig_depth.show()

### 3. Thuật toán tách câu bảo vệ ngữ cảnh học thuật và xử lý phiên bản rỗng

Đề bài nêu rõ thách thức lớn nhất khi tách lá câu: *"do not split inside mathematics, commands, abbreviations such as i.e. and et al. and Fig. 1, URLs, or decimal numbers, since a dot that does not end a sentence does not create a new leaf."*

Module tách câu sử dụng giải thuật **Che chắn trước — Tách câu sau — Phục hồi nguyên bản (Masking-based splitting)**:
1. **Bảo vệ biểu thức toán inline:** Thay thế toàn bộ `$[^$]+$` bằng token an toàn `\x00MATH_i\x00`.
2. **Bảo vệ số thập phân:** Nhận diện regex `\b\d+\.\d+\b` (tránh cắt nhầm số `3.14`).
3. **Bảo vệ từ viết tắt học thuật:** Nhận diện danh mục viết tắt `i.e.`, `e.g.`, `et al.`, `Fig.`, `Tab.`, `Eq.`, `Sec.`, `Ref.`.
4. **Tách câu:** Chỉ ngắt tại dấu chấm theo sau bởi khoảng trắng và chữ hoa: `(?<=[.!?])\s+(?=[A-Z\\(])`.
5. **Phục hồi:** Điền ngược nội dung đã che chắn vào các câu đã tách.

#### Quy chuẩn bắt buộc xử lý phiên bản rỗng (Empty Versions)
Đề bài nhấn mạnh: Không được bỏ sót bất kỳ phiên bản nào của bài báo. Khi gặp bài báo nộp dạng PDF scan hoặc mã nguồn TeX bị hỏng không thể biên dịch:
- Không sinh exception hay bỏ qua phiên bản.
- Trả về đúng **1 nút rễ rỗng** (`kind="document"`, `content=""`) với `status = "empty"`.
- Ghi nhận chính xác 1.022 phiên bản rỗng trong `parse.csv` (chiếm 6,50%), duy trì tính toàn vẹn của chuỗi lịch sử phiên bản.

In [4]:
# Kiểm thử thực tế giải thuật tách câu bảo vệ ngữ cảnh học thuật
import re

PROTECT_RULES = [
    (re.compile(r"\$[^$]+\$"), "MATH"),
    (re.compile(r"https?://\S+"), "URL"),
    (re.compile(r"\b\d+\.\d+\b"), "NUM"),
    (re.compile(r"\b(?:i\.e|e\.g|et al|cf|vs|resp|approx|Fig|Eq|Sec|Tab|Ref|No|Dr|Prof|Mr|Ms|St|al)\.", re.I), "ABBR"),
]

def safe_split_sentences(text: str) -> list[str]:
    store = []
    def mask(m):
        store.append(m.group(0))
        return f"\x00{len(store)-1}\x00"
    for pat, _ in PROTECT_RULES:
        text = pat.sub(mask, text)
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z\\(])", text)
    out = []
    for p in parts:
        p = re.sub(r"\x00(\d+)\x00", lambda m: store[int(m.group(1))], p).strip()
        if p:
            out.append(p)
    return out

test_passage = "We evaluate BERT (Devlin et al. 2019) on SQuAD. The accuracy improved by 3.14 times with $O(N^2)$ attention. See Fig. 2 for details."
sentences = safe_split_sentences(test_passage)

print("KIỂM ĐỊNH KỸ THUẬT TÁCH CÂU HỌC THUẬT:")
print(f"- Đoạn văn đầu vào : {test_passage}")
print(f"- Số câu bóc tách  : {len(sentences)} câu (Kỳ vọng: 3 câu, không bị ngắt ở 'et al.', '3.14' hay 'Fig. 2')")
for idx, s in enumerate(sentences, 1):
    print(f"  [{idx}] {s}")

print("\nMINH HỌA CẤU TRÚC BẢNG ELEMENTS VÀ HIERARCHY MẪU:")
if df_elem_sample is not None:
    print("--- Bảng elements (Kho phần tử dùng chung, 5 dòng đầu) ---")
    display(df_elem_sample.head(5))
if df_hier_sample is not None:
    print("--- Bảng hierarchy (Quan hệ cây phân cấp, 5 dòng đầu) ---")
    display(df_hier_sample.head(5))

KIỂM ĐỊNH KỸ THUẬT TÁCH CÂU HỌC THUẬT:
- Đoạn văn đầu vào : We evaluate BERT (Devlin et al. 2019) on SQuAD. The accuracy improved by 3.14 times with $O(N^2)$ attention. See Fig. 2 for details.
- Số câu bóc tách  : 3 câu (Kỳ vọng: 3 câu, không bị ngắt ở 'et al.', '3.14' hay 'Fig. 2')
  [1] We evaluate BERT (Devlin et al. 2019) on SQuAD.
  [2] The accuracy improved by 3.14 times with $O(N^2)$ attention.
  [3] See Fig. 2 for details.

MINH HỌA CẤU TRÚC BẢNG ELEMENTS VÀ HIERARCHY MẪU:
--- Bảng elements (Kho phần tử dùng chung, 5 dòng đầu) ---


,element_id,arxiv_id,kind,content
0,2507-06659#doc_fb30f193,2507.06659,document,2507.06659v1
1,2507-06659#snt_9a73ab3c,2507.06659,sentence,\title{Modeling Multistability and Hysteresis ...
2,2507-06659#snt_94169bcf,2507.06659,sentence,Growing evidence suggests that the macroscopic...
3,2507-06659#snt_0bbfa21c,2507.06659,sentence,"Here, we demonstrate that in real-world road n..."
4,2507-06659#snt_eaa5fe42,2507.06659,sentence,"By analyzing real-world urban traffic data, we..."


--- Bảng hierarchy (Quan hệ cây phân cấp, 5 dòng đầu) ---


,arxiv_id,version,element_id,parent_id
0,2507.06659,1,2507-06659#snt_9a73ab3c,2507-06659#doc_fb30f193
1,2507.06659,1,2507-06659#snt_94169bcf,2507-06659#doc_fb30f193
2,2507.06659,1,2507-06659#snt_0bbfa21c,2507-06659#doc_fb30f193
3,2507.06659,1,2507-06659#snt_eaa5fe42,2507-06659#doc_fb30f193
4,2507.06659,1,2507-06659#snt_b55dd4ce,2507-06659#doc_fb30f193
